# ObstacleTrack - Ablation v4 (statistically usable Table 3)

Attach **RoadObstacle21 Zip**, enable **GPU T4 x2**, then **Run All** once.

## What changed and why

The previous single-seed run was not publishable for three reasons:

1. **Temperature scaling cannot change AuPRC / FPR@95 / IoU.** Dividing logits by a positive
   constant is strictly monotone, so it cannot reorder any pixel scores. That is why D was
   identical to C and F was identical to B. Reporting TS as improving those metrics would be a
   mathematical error. TS is now evaluated **only** on NLL, ECE-15 and Brier.
2. **No variance estimate.** Config A scored 68.62 in one run and 57.87 in the next under the same
   seed but a different execution setup. Effects smaller than that swing cannot be claimed from one run.
3. **Component detection was 3/6 in every configuration** - it has no discriminative power at n=6.

## This version

- **5 seeds** (42, 1, 2, 3, 4). Each seed redraws **both** the train/validation split and the head
  initialisation, so the 6-image split is no longer a single arbitrary draw.
- Reports **mean +/- SD**, **paired t-tests on shared seeds**, **95% CIs** and **Cohen's dz**.
- Prints an explicit *significant / NOT significant* verdict per contrast.
- Emits **paste-ready LaTeX** for Table 3 plus a separate calibration table.
- Same stability design as before: one fresh subprocess per run, single T4, `num_workers=0`,
   atomic writes, 20-minute watchdog, one retry, full resume on re-run.

20 trainings at roughly 1.2-2 min each: **about 30-45 minutes**. A failed cell can simply be re-run;
completed work is skipped.


**v4 change:** the SegFormer backbone is downloaded once in the setup cell and all worker subprocesses load it from local disk fully offline (`HF_HUB_OFFLINE=1`). This removes the HF-hub rate-limit stall that froze config C in v3. Training watchdog raised to 30 min. Per-epoch wall-clock added to logs. Results/checkpoint paths are unchanged, so any completed (config, seed) results are still skipped on resume.

## 1. Write the isolated worker


In [1]:
from pathlib import Path

WORKER_PATH = Path("/kaggle/working/worker_v4.py")
WORKER_SOURCE = 'import os, sys, gc, json, time, random, argparse, warnings\nfrom pathlib import Path\n\nos.environ.setdefault("TOKENIZERS_PARALLELISM", "false")\nwarnings.filterwarnings("ignore")\n\nimport cv2\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport albumentations as A\nfrom scipy.ndimage import label as scipy_label\nfrom sklearn.metrics import average_precision_score, roc_curve\nfrom sklearn.model_selection import train_test_split\nfrom torch.utils.data import Dataset, DataLoader\nfrom transformers import SegformerForSemanticSegmentation\n\nMODEL_NAME = os.environ.get("OBSTACLETRACK_MODEL_DIR",\n    "nvidia/segformer-b0-finetuned-ade-512-512")  # local dir => no network, no HF-hub stalls\nH, W = 576, 1024\nBATCH_SIZE = 2\nGRAD_ACCUM = 4\nEPOCHS = 12\nPATIENCE = 4\nLR = 3e-4\nWEIGHT_DECAY = 1e-2\nDICE_WEIGHT = 0.5\nSEP_WEIGHT = 0.1\nCOPY_PASTE_PROB = 0.5\nIGNORE = 255\n\nROOT = Path("/kaggle/working/ablation_v3")\nRESULTS_DIR = ROOT / "results"\nCKPT_DIR = ROOT / "checkpoints"\nfor d in (ROOT, RESULTS_DIR, CKPT_DIR):\n    d.mkdir(parents=True, exist_ok=True)\n\n# train=False configs reuse another config\'s checkpoint: temperature scaling is a\n# post-hoc transform and must never retrain the network.\nCONFIGS = {\n    "A": dict(name="Fine-tuned SegFormer-B0 (CE+Dice)", cp=False, sep=False, ts=False, train=True, source=None),\n    "B": dict(name="+ Perspective-aware copy-paste", cp=True, sep=False, ts=False, train=True, source=None),\n    "C": dict(name="+ Separation loss", cp=True, sep=True, ts=False, train=True, source=None),\n    "D": dict(name="+ Temperature scaling (full ObstacleTrack)", cp=True, sep=True, ts=True, train=False, source="C"),\n    "E": dict(name="Full minus copy-paste", cp=False, sep=True, ts=True, train=True, source=None),\n    "F": dict(name="Full minus separation loss", cp=True, sep=False, ts=True, train=False, source="B"),\n}\n\n\ndef seed_everything(seed):\n    random.seed(seed)\n    os.environ["PYTHONHASHSEED"] = str(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False\n\n\ndef find_dataset_root():\n    cands = []\n    for p in Path("/kaggle/input").rglob("dataset_ObstacleTrack"):\n        if (p / "images").is_dir() and (p / "labels_masks").is_dir():\n            cands.append(p)\n    if not cands:\n        for p in Path("/kaggle/input").rglob("*"):\n            if p.is_dir() and (p / "images").is_dir() and (p / "labels_masks").is_dir():\n                cands.append(p)\n    if not cands:\n        raise FileNotFoundError("dataset_ObstacleTrack not found under /kaggle/input")\n    return sorted(cands, key=lambda x: len(str(x)))[0]\n\n\ndef clean_stem(name):\n    stem = Path(name).stem\n    for suf in ["_labels_semantic", "_label", "_mask", "_gt", "_seg", "_gtFine_labelIds"]:\n        stem = stem.replace(suf, "")\n    return stem.lower().strip("_- ")\n\n\ndef discover_pairs(root):\n    exts = {".png", ".jpg", ".jpeg", ".webp"}\n    imgs = [p for p in sorted((root / "images").rglob("*")) if p.is_file() and p.suffix.lower() in exts]\n    msks = [p for p in sorted((root / "labels_masks").rglob("*")) if p.is_file() and p.suffix.lower() in exts]\n    by = {}\n    for m in msks:\n        by.setdefault(clean_stem(m.name), []).append(m)\n    pairs = [(i, by[clean_stem(i.name)][0]) for i in imgs if clean_stem(i.name) in by]\n    if not pairs:\n        raise RuntimeError("no labeled pairs found")\n    return pairs\n\n\ndef remap_mask(mask):\n    out = np.full(mask.shape, 255, dtype=np.uint8)\n    out[mask == 0] = 0\n    out[(mask == 1) | (mask == 2)] = 1\n    out[mask == 255] = 255\n    return out\n\n\nclass ObstacleBank:\n    def __init__(self, pairs, max_patches=300):\n        self.patches = []\n        for ip, mp in pairs:\n            img = cv2.cvtColor(cv2.imread(str(ip)), cv2.COLOR_BGR2RGB)\n            raw = cv2.imread(str(mp), cv2.IMREAD_UNCHANGED)\n            if raw.ndim == 3:\n                raw = raw[:, :, 0]\n            obs = (remap_mask(raw) == 1).astype(np.uint8)\n            cnts, _ = cv2.findContours(obs, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)\n            for c in cnts:\n                x, y, w, h = cv2.boundingRect(c)\n                if w < 10 or h < 10:\n                    continue\n                rgba = np.zeros((h, w, 4), np.uint8)\n                rgba[:, :, :3] = img[y:y+h, x:x+w]\n                rgba[:, :, 3] = obs[y:y+h, x:x+w] * 255\n                self.patches.append((rgba, y + h / 2.0, img.shape[0], img.shape[1]))\n                if len(self.patches) >= max_patches:\n                    break\n            if len(self.patches) >= max_patches:\n                break\n        print(f"obstacle bank: {len(self.patches)} patches", flush=True)\n\n    def paste_onto(self, img, mask, p=COPY_PASTE_PROB):\n        if random.random() > p or not self.patches:\n            return img, mask\n        rgba, sy, sh, sw = random.choice(self.patches)\n        ph, pw = rgba.shape[:2]\n        ty = random.randint(img.shape[0] // 3, img.shape[0] - 10)\n        scale = (0.3 + 0.7 * (ty / img.shape[0])) * min(min(img.shape[0] / max(sh, 1), img.shape[1] / max(sw, 1)), 1.0)\n        nh = min(max(int(ph * scale), 10), img.shape[0] // 2)\n        nw = min(max(int(pw * scale), 10), img.shape[1] // 2)\n        r = cv2.resize(rgba, (nw, nh), interpolation=cv2.INTER_LINEAR)\n        rgb, alpha = r[:, :, :3], r[:, :, 3].astype(np.float32) / 255.0\n        ys = max(0, ty - nh // 2)\n        xs = random.randint(0, max(0, img.shape[1] - nw))\n        ye, xe = min(ys + nh, img.shape[0]), min(xs + nw, img.shape[1])\n        ch, cw = ye - ys, xe - xs\n        a = alpha[:ch, :cw, None]\n        img[ys:ye, xs:xe] = (img[ys:ye, xs:xe] * (1 - a) + rgb[:ch, :cw] * a).astype(np.uint8)\n        mask[ys:ye, xs:xe] = np.where(alpha[:ch, :cw] > 0.5, 1, mask[ys:ye, xs:xe])\n        return img, mask\n\n\nclass ObstacleDataset(Dataset):\n    def __init__(self, pairs, augment=False, bank=None, cp=False):\n        self.pairs, self.augment, self.bank, self.cp = pairs, augment, bank, cp\n        self.tf = A.Compose([\n            A.HorizontalFlip(p=0.5),\n            A.RandomBrightnessContrast(p=0.3),\n            A.GaussNoise(std_range=(0.01, 0.05), p=0.2),\n            A.RandomGamma(p=0.2),\n        ]) if augment else None\n        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)\n        self.std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)\n\n    def __len__(self):\n        return len(self.pairs)\n\n    def __getitem__(self, i):\n        ip, mp = self.pairs[i]\n        img = cv2.cvtColor(cv2.imread(str(ip)), cv2.COLOR_BGR2RGB)\n        raw = cv2.imread(str(mp), cv2.IMREAD_UNCHANGED)\n        if raw.ndim == 3:\n            raw = raw[:, :, 0]\n        mask = remap_mask(raw)\n        img = cv2.resize(img, (W, H), interpolation=cv2.INTER_LINEAR)\n        mask = cv2.resize(mask, (W, H), interpolation=cv2.INTER_NEAREST)\n        if self.augment and self.cp and self.bank is not None:\n            img, mask = self.bank.paste_onto(img.copy(), mask.copy())\n        if self.tf is not None:\n            z = self.tf(image=img, mask=mask)\n            img, mask = z["image"], z["mask"]\n        x = torch.from_numpy(np.ascontiguousarray(img)).permute(2, 0, 1).float() / 255.0\n        return (x - self.mean) / self.std, torch.from_numpy(np.ascontiguousarray(mask)).long()\n\n\nclass DiceLoss(nn.Module):\n    def forward(self, logits, t):\n        p = torch.sigmoid(logits[:, 1])\n        v = t != IGNORE\n        pv, tv = p[v], t[v].float()\n        return 1.0 - (2.0 * (pv * tv).sum() + 1.0) / (pv.sum() + tv.sum() + 1.0)\n\n\nclass SeparationLoss(nn.Module):\n    def __init__(self, margin=0.5):\n        super().__init__()\n        self.margin = margin\n\n    def forward(self, logits, t):\n        p = torch.sigmoid(logits[:, 1])\n        v = t != IGNORE\n        obs, bg = (t == 1) & v, (t == 0) & v\n        parts = []\n        if obs.any():\n            parts.append(F.relu(self.margin - p[obs]).mean())\n        if bg.any():\n            parts.append(F.relu(p[bg] - (1 - self.margin)).mean())\n        return torch.stack(parts).mean() if parts else logits.sum() * 0.0\n\n\nclass CombinedLoss(nn.Module):\n    def __init__(self, use_sep):\n        super().__init__()\n        self.ce = nn.CrossEntropyLoss(ignore_index=IGNORE)\n        self.dice = DiceLoss()\n        self.sep = SeparationLoss() if use_sep else None\n\n    def forward(self, logits, t):\n        loss = self.ce(logits, t) + DICE_WEIGHT * self.dice(logits, t)\n        if self.sep is not None:\n            loss = loss + SEP_WEIGHT * self.sep(logits, t)\n        return loss\n\n\nclass ObstacleSegFormer(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.backbone = SegformerForSemanticSegmentation.from_pretrained(\n            MODEL_NAME, num_labels=2, ignore_mismatched_sizes=True)\n\n    def forward(self, x):\n        return self.backbone(pixel_values=x).logits\n\n\ndef atomic_json(path, obj):\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    tmp.write_text(json.dumps(obj, indent=2))\n    os.replace(tmp, path)\n\n\ndef train_model(cfg, seed, train_pairs, val_pairs, ckpt_path):\n    seed_everything(seed)\n    bank = ObstacleBank(train_pairs) if cfg["cp"] else None\n    tr = ObstacleDataset(train_pairs, augment=True, bank=bank, cp=cfg["cp"])\n    va = ObstacleDataset(val_pairs, augment=False)\n    g = torch.Generator().manual_seed(seed)\n    trl = DataLoader(tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False, drop_last=True, generator=g)\n    val = DataLoader(va, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)\n    dev = torch.device("cuda:0")\n    model = ObstacleSegFormer().to(dev)\n    lf = CombinedLoss(cfg["sep"])\n    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)\n    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)\n    sc = torch.amp.GradScaler("cuda", enabled=True)\n    best, bad = float("inf"), 0\n    for ep in range(EPOCHS):\n        t_ep = time.time()\n        model.train()\n        opt.zero_grad(set_to_none=True)\n        for i, (x, y) in enumerate(trl):\n            x, y = x.to(dev), y.to(dev)\n            with torch.amp.autocast("cuda", enabled=True):\n                lo = F.interpolate(model(x), size=y.shape[-2:], mode="bilinear", align_corners=False)\n                loss = lf(lo, y) / GRAD_ACCUM\n            sc.scale(loss).backward()\n            if (i + 1) % GRAD_ACCUM == 0:\n                sc.step(opt)\n                sc.update()\n                opt.zero_grad(set_to_none=True)\n        sch.step()\n        model.eval()\n        tot = n = 0\n        with torch.inference_mode():\n            for x, y in val:\n                x, y = x.to(dev), y.to(dev)\n                with torch.amp.autocast("cuda", enabled=True):\n                    lo = F.interpolate(model(x), size=y.shape[-2:], mode="bilinear", align_corners=False)\n                    tot += lf(lo, y).item()\n                    n += 1\n        vl = tot / max(n, 1)\n        print(f"  epoch {ep+1:02d}/{EPOCHS} val_loss={vl:.6f} ({time.time()-t_ep:.0f}s)", flush=True)\n        if vl < best - 1e-4:\n            best, bad = vl, 0\n            torch.save({"state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},\n                        "best_val_loss": best, "epoch": ep + 1}, ckpt_path)\n        else:\n            bad += 1\n            if bad >= PATIENCE:\n                print(f"  early stop @ epoch {ep+1}", flush=True)\n                break\n    del model, opt, sch, sc, trl, val, tr, va, bank\n    gc.collect()\n    torch.cuda.empty_cache()\n\n\ndef load_input(ip, mp):\n    img = cv2.resize(cv2.cvtColor(cv2.imread(str(ip)), cv2.COLOR_BGR2RGB), (W, H), interpolation=cv2.INTER_LINEAR)\n    raw = cv2.imread(str(mp), cv2.IMREAD_UNCHANGED)\n    if raw.ndim == 3:\n        raw = raw[:, :, 0]\n    gt = cv2.resize(remap_mask(raw), (W, H), interpolation=cv2.INTER_NEAREST)\n    x = img.astype(np.float32).transpose(2, 0, 1) / 255.0\n    x = (x - np.array([0.485, 0.456, 0.406], np.float32)[:, None, None]) / np.array([0.229, 0.224, 0.225], np.float32)[:, None, None]\n    return torch.from_numpy(x).unsqueeze(0), gt\n\n\ndef collect_logits(ckpt_path, pairs):\n    dev = torch.device("cuda:0")\n    model = ObstacleSegFormer().to(dev)\n    obj = torch.load(ckpt_path, map_location="cpu")\n    model.load_state_dict(obj["state_dict"])\n    model.eval()\n    out = []\n    with torch.inference_mode():\n        for ip, mp in pairs:\n            x, gt = load_input(ip, mp)\n            with torch.amp.autocast("cuda", enabled=True):\n                lo = F.interpolate(model(x.to(dev)), size=gt.shape, mode="bilinear", align_corners=False)\n            # store the binary decision margin only: halves memory, loses nothing\n            z = (lo[0, 1] - lo[0, 0]).float().cpu().numpy().astype(np.float16)\n            out.append((z, gt, Path(ip).name))\n    del model, obj\n    gc.collect()\n    torch.cuda.empty_cache()\n    return out\n\n\ndef sigmoid(z):\n    o = np.empty_like(z, dtype=np.float32)\n    p = z >= 0\n    o[p] = 1.0 / (1.0 + np.exp(-z[p]))\n    e = np.exp(z[~p])\n    o[~p] = e / (1.0 + e)\n    return o\n\n\ndef nll_at(pairs, temp):\n    tot = cnt = 0\n    for z16, gt, _ in pairs:\n        v = gt != IGNORE\n        z = z16.astype(np.float32)[v] / temp\n        y = gt[v].astype(np.float32)\n        tot += float(np.sum(np.logaddexp(0.0, z) - y * z, dtype=np.float64))\n        cnt += y.size\n    return tot / max(cnt, 1)\n\n\ndef fit_temperature(pairs):\n    ts = np.round(np.arange(0.5, 5.01, 0.1), 2)\n    vals = [nll_at(pairs, float(t)) for t in ts]\n    i = int(np.argmin(vals))\n    return float(ts[i]), float(vals[i])\n\n\ndef calibration(pairs, temp, bins=15):\n    cnt = np.zeros(bins, np.int64)\n    conf = np.zeros(bins, np.float64)\n    acc = np.zeros(bins, np.float64)\n    brier = 0.0\n    n = 0\n    for z16, gt, _ in pairs:\n        v = gt != IGNORE\n        p = sigmoid(z16.astype(np.float32)[v] / temp)\n        y = gt[v].astype(np.float32)\n        pred = (p >= 0.5).astype(np.float32)\n        c = np.where(pred == 1, p, 1.0 - p)\n        ok = (pred == y).astype(np.float32)\n        idx = np.minimum((c * bins).astype(np.int32), bins - 1)\n        np.add.at(cnt, idx, 1)\n        np.add.at(conf, idx, c)\n        np.add.at(acc, idx, ok)\n        brier += float(np.sum((p - y) ** 2, dtype=np.float64))\n        n += y.size\n    ece = sum((cnt[b] / n) * abs(acc[b] / cnt[b] - conf[b] / cnt[b]) for b in range(bins) if cnt[b])\n    return {"nll": float(nll_at(pairs, temp)), "ece15": float(ece), "brier": float(brier / max(n, 1))}\n\n\ndef evaluate(pairs, temp):\n    per_image = []\n    det = comp = 0\n    for z16, gt, name in pairs:\n        prob = sigmoid(z16.astype(np.float32) / temp)\n        v = gt != IGNORE\n        y, p = gt[v].astype(np.int32), prob[v].astype(np.float64)\n        rec = {"image": name}\n        if np.unique(y).size >= 2:\n            fpr, tpr, _ = roc_curve(y, p)\n            j = min(int(np.searchsorted(tpr, 0.95)), len(fpr) - 1)\n            pred = p >= 0.5\n            rec.update({\n                "auprc": 100.0 * float(average_precision_score(y, p)),\n                "fpr95": 100.0 * float(fpr[j]),\n                "iou_05": 100.0 * float(np.sum(pred & (y == 1)) / max(np.sum(pred | (y == 1)), 1)),\n            })\n        lab, k = scipy_label((gt == 1).astype(np.uint8))\n        d = sum(1 for c in range(1, k + 1) if (prob >= 0.5)[lab == c].sum() > 0.5 * (lab == c).sum())\n        rec.update({"components": int(k), "detected": int(d)})\n        comp += k\n        det += d\n        per_image.append(rec)\n    sc = [r for r in per_image if "auprc" in r]\n    return {\n        "auprc": float(np.mean([r["auprc"] for r in sc])),\n        "fpr95": float(np.mean([r["fpr95"] for r in sc])),\n        "iou_05": float(np.mean([r["iou_05"] for r in sc])),\n        "comp_det": 100.0 * det / max(comp, 1),\n        "n_images": len(sc),\n        "n_components": int(comp),\n        "detected_components": int(det),\n        "per_image": per_image,\n    }\n\n\ndef main(tag, seed):\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA unavailable; enable the Kaggle GPU accelerator")\n    cfg = CONFIGS[tag]\n    seed_everything(seed)\n    pairs = discover_pairs(find_dataset_root())\n    # The split varies with the seed: with only 6 validation images per run, varying\n    # the split is what lets 5 seeds cover most of the 30 labelled frames.\n    train_pairs, val_pairs = train_test_split(pairs, test_size=0.2, random_state=seed)\n    sdir = RESULTS_DIR / f"seed_{seed}"\n    cdir = CKPT_DIR / f"seed_{seed}"\n    sdir.mkdir(parents=True, exist_ok=True)\n    cdir.mkdir(parents=True, exist_ok=True)\n    src = cfg["source"] or tag\n    ckpt = cdir / f"{src}.pt"\n    print(f"TAG={tag} SEED={seed} | {cfg[\'name\']}", flush=True)\n    t0 = time.time()\n    if cfg["train"]:\n        train_model(cfg, seed, train_pairs, val_pairs, ckpt)\n    elif not ckpt.exists():\n        raise FileNotFoundError(f"{tag} needs checkpoint {ckpt}; run {src} first")\n    lp = collect_logits(ckpt, val_pairs)\n    pre = calibration(lp, 1.0)\n    temp = fit_temperature(lp)[0] if cfg["ts"] else 1.0\n    post = calibration(lp, temp)\n    met = evaluate(lp, temp)\n    res = {"tag": tag, "seed": seed, "name": cfg["name"], "cp": cfg["cp"], "sep": cfg["sep"], "ts": cfg["ts"],\n           "temperature": temp, **met, "calibration_before": pre, "calibration_after": post,\n           "checkpoint_source": src, "minutes": round((time.time() - t0) / 60.0, 2)}\n    atomic_json(sdir / f"result_{tag}.json", res)\n    print(f"OK {tag} seed={seed} auprc={met[\'auprc\']:.2f} fpr95={met[\'fpr95\']:.3f} "\n          f"iou={met[\'iou_05\']:.2f} T={temp} nll {pre[\'nll\']:.5f}->{post[\'nll\']:.5f}", flush=True)\n    del lp\n    gc.collect()\n    torch.cuda.empty_cache()\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--tag", required=True, choices=list(CONFIGS))\n    ap.add_argument("--seed", required=True, type=int)\n    main(ap.parse_args().tag, ap.parse_args().seed)\n'
WORKER_PATH.write_text(WORKER_SOURCE, encoding="utf-8")
print(f"Worker ready: {WORKER_PATH} ({len(WORKER_SOURCE.splitlines())} lines)")

# Download the backbone EXACTLY ONCE, up front. Every worker subprocess then
# loads it from local disk fully offline. This fixes the v3 failure mode where
# unauthenticated HF-hub requests from 20 fresh subprocesses got rate-limited
# and one from_pretrained() call stalled until the watchdog killed the run.
LOCAL_MODEL = Path("/kaggle/working/segformer_b0_local")
if not (LOCAL_MODEL / "config.json").exists():
    print("downloading nvidia/segformer-b0-finetuned-ade-512-512 (one time) ...")
    from transformers import SegformerForSemanticSegmentation
    _m = SegformerForSemanticSegmentation.from_pretrained("nvidia/segformer-b0-finetuned-ade-512-512")
    _m.save_pretrained(LOCAL_MODEL)
    del _m
print("Local backbone ready:", LOCAL_MODEL)


Worker ready: /kaggle/working/worker_v4.py (450 lines)
downloading nvidia/segformer-b0-finetuned-ade-512-512 (one time) ...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/15.0M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Local backbone ready: /kaggle/working/segformer_b0_local


## 2. Run 5 seeds x 6 configs, aggregate, test significance, emit LaTeX


In [2]:
import os, sys, gc, json, time, shutil, subprocess, itertools
from pathlib import Path
import numpy as np
from scipy import stats
from IPython.display import FileLink, display

ROOT = Path("/kaggle/working/ablation_v3")
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
WORKER = Path("/kaggle/working/worker_v4.py")
LOCAL_MODEL = Path("/kaggle/working/segformer_b0_local")

SEEDS = [42, 1, 2, 3, 4]
ORDER = ["A", "B", "C", "D", "E", "F"]
TRAINING = {"A", "B", "C", "E"}
NAMES = {
    "A": "Fine-tuned SegFormer-B0 (CE+Dice)",
    "B": "+ Perspective-aware copy-paste",
    "C": "+ Separation loss",
    "D": "+ Temperature scaling (full)",
    "E": "Full minus copy-paste",
    "F": "Full minus separation loss",
}
NEEDED = {"tag", "seed", "auprc", "fpr95", "iou_05", "temperature", "calibration_before", "calibration_after"}


def path_for(tag, seed):
    return RESULTS_DIR / f"seed_{seed}" / f"result_{tag}.json"


def ok(tag, seed):
    p = path_for(tag, seed)
    if not p.exists():
        return False
    try:
        o = json.loads(p.read_text())
        return o.get("tag") == tag and o.get("seed") == seed and NEEDED.issubset(o)
    except Exception:
        return False


def load(tag, seed):
    return json.loads(path_for(tag, seed).read_text())


def run(tag, seed, attempt):
    timeout = 30 * 60 if tag in TRAINING else 10 * 60
    env = os.environ.copy()
    env.update({"CUDA_VISIBLE_DEVICES": "0", "PYTHONUNBUFFERED": "1",
                "TOKENIZERS_PARALLELISM": "false", "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2",
                "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:128",
                "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
                "OBSTACLETRACK_MODEL_DIR": str(LOCAL_MODEL)})
    print(f"\n--- {tag} seed={seed} (attempt {attempt}/2, timeout {timeout//60}m) ---", flush=True)
    try:
        r = subprocess.run([sys.executable, "-u", str(WORKER), "--tag", tag, "--seed", str(seed)],
                           env=env, timeout=timeout, check=False)
        return r.returncode == 0 and ok(tag, seed)
    except subprocess.TimeoutExpired:
        print(f"WATCHDOG killed {tag} seed={seed}", flush=True)
        return False


if not WORKER.exists():
    raise FileNotFoundError("Worker missing - run the setup cell first.")
if not (LOCAL_MODEL / "config.json").exists():
    raise FileNotFoundError("Local backbone missing - run the setup cell first (it downloads the model once).")

print(f"Seeds: {SEEDS}   Configs: {ORDER}")
print("Each (config, seed) runs in a fresh process. D reuses C, F reuses B.")
print(f"Trainings: {len(SEEDS) * len(TRAINING)}  |  expect roughly 40-160 min depending on session speed\n")

failures = []
for seed in SEEDS:
    for tag in ORDER:
        if ok(tag, seed):
            print(f"skip {tag} seed={seed} (already done)", flush=True)
            continue
        done = False
        for attempt in (1, 2):
            done = run(tag, seed, attempt)
            gc.collect()
            time.sleep(5)
            if done:
                break
        if not done:
            failures.append((tag, seed))
            print(f"FAILED {tag} seed={seed} - continuing", flush=True)

# ---------------- aggregation ----------------
rows = {t: [load(t, s) for s in SEEDS if ok(t, s)] for t in ORDER}
complete = [t for t in ORDER if len(rows[t]) == len(SEEDS)]
print(f"\nComplete configs: {complete}   Failures: {failures or 'none'}")

summary = {}
for t in ORDER:
    if not rows[t]:
        continue
    g = lambda k: np.array([r[k] for r in rows[t]], float)
    nb = np.array([r["calibration_before"]["nll"] for r in rows[t]], float)
    na = np.array([r["calibration_after"]["nll"] for r in rows[t]], float)
    eb = np.array([r["calibration_before"]["ece15"] for r in rows[t]], float)
    ea = np.array([r["calibration_after"]["ece15"] for r in rows[t]], float)
    summary[t] = {
        "tag": t, "name": NAMES[t], "n_seeds": len(rows[t]),
        "cp": rows[t][0]["cp"], "sep": rows[t][0]["sep"], "ts": rows[t][0]["ts"],
        "auprc": [g("auprc").mean(), g("auprc").std(ddof=1)],
        "fpr95": [g("fpr95").mean(), g("fpr95").std(ddof=1)],
        "iou_05": [g("iou_05").mean(), g("iou_05").std(ddof=1)],
        "comp_det": [g("comp_det").mean(), g("comp_det").std(ddof=1)],
        "temperature": [g("temperature").mean(), g("temperature").std(ddof=1)],
        "nll_before": [nb.mean(), nb.std(ddof=1)], "nll_after": [na.mean(), na.std(ddof=1)],
        "ece_before": [eb.mean(), eb.std(ddof=1)], "ece_after": [ea.mean(), ea.std(ddof=1)],
        "per_seed_auprc": g("auprc").tolist(),
    }


def paired(a, b, key, sub=None):
    """Paired comparison of config a vs b across shared seeds."""
    seeds = [s for s in SEEDS if ok(a, s) and ok(b, s)]
    if len(seeds) < 3:
        return None
    va = np.array([(load(a, s)[sub][key] if sub else load(a, s)[key]) for s in seeds], float)
    vb = np.array([(load(b, s)[sub][key] if sub else load(b, s)[key]) for s in seeds], float)
    d = va - vb
    if np.allclose(d, 0):
        return {"contrast": f"{a} - {b}", "metric": key, "n": len(seeds), "mean_diff": 0.0,
                "sd_diff": 0.0, "t": None, "p": 1.0, "cohen_dz": 0.0, "ci95": [0.0, 0.0],
                "note": "identical by construction"}
    t, p = stats.ttest_rel(va, vb)
    sd = d.std(ddof=1)
    se = sd / np.sqrt(len(d))
    crit = stats.t.ppf(0.975, len(d) - 1)
    return {"contrast": f"{a} - {b}", "metric": key, "n": len(seeds),
            "mean_diff": float(d.mean()), "sd_diff": float(sd), "t": float(t), "p": float(p),
            "cohen_dz": float(d.mean() / sd) if sd > 0 else None,
            "ci95": [float(d.mean() - crit * se), float(d.mean() + crit * se)]}


contrasts = []
for a, b, label in [("B", "A", "copy-paste added to baseline"),
                    ("C", "B", "separation loss added"),
                    ("C", "E", "copy-paste removed from full (leave-one-out)"),
                    ("C", "F", "separation loss removed from full (leave-one-out)")]:
    for m in ("auprc", "fpr95", "iou_05"):
        r = paired(a, b, m)
        if r:
            r["label"] = label
            contrasts.append(r)

# temperature scaling: calibration only (rank metrics are invariant to a monotone rescale)
ts_stats = []
for t in ("D", "E", "F"):
    seeds = [s for s in SEEDS if ok(t, s)]
    if len(seeds) < 3:
        continue
    for key, nice in (("nll", "NLL"), ("ece15", "ECE-15"), ("brier", "Brier")):
        before = np.array([load(t, s)["calibration_before"][key] for s in seeds], float)
        after = np.array([load(t, s)["calibration_after"][key] for s in seeds], float)
        d = after - before
        tt, p = stats.ttest_rel(after, before)
        ts_stats.append({"config": t, "metric": nice, "n": len(seeds),
                         "before": float(before.mean()), "after": float(after.mean()),
                         "mean_diff": float(d.mean()), "p": float(p),
                         "rel_change_pct": float(100.0 * d.mean() / before.mean())})

out = {"seeds": SEEDS, "summary": summary, "contrasts": contrasts,
       "temperature_scaling": ts_stats, "failures": failures,
       "protocol": "single T4; batch 2 x grad_accum 4; workers=0; split and init both vary with seed; "
                   "per-image mean metrics on the 6-image validation split; mean +/- SD over seeds"}
tmp = ROOT / "ablation_summary.json.tmp"
tmp.write_text(json.dumps(out, indent=2))
os.replace(tmp, ROOT / "ablation_summary.json")

all_rows = [r for t in ORDER for r in rows[t]]
for r in all_rows:
    r.pop("per_image", None)
(ROOT / "ablation_results_all_seeds.json").write_text(json.dumps(all_rows, indent=2))

# ---------------- report ----------------
W = 108
print("\n" + "=" * W)
print(f"TABLE 3 - ABLATION, mean +/- SD over {len(SEEDS)} seeds {SEEDS}")
print("=" * W)
print(f"{'':3} {'CP':<3}{'Sep':<4}{'TS':<4} {'AuPRC':>14} {'FPR@95':>14} {'IoU@0.5':>14} {'CompDet':>13}")
for t in ORDER:
    if t not in summary:
        continue
    s = summary[t]
    f = lambda k, d=2: f"{s[k][0]:.{d}f}+/-{s[k][1]:.{d}f}"
    print(f"{t:<3} {'Y' if s['cp'] else '-':<3}{'Y' if s['sep'] else '-':<4}{'Y' if s['ts'] else '-':<4} "
          f"{f('auprc'):>14} {f('fpr95',3):>14} {f('iou_05'):>14} {f('comp_det'):>13}")

print("\n" + "=" * W)
print("CALIBRATION - the only place temperature scaling can legitimately appear")
print("=" * W)
print(f"{'':3} {'T':>12} {'NLL before':>12} {'NLL after':>12} {'ECE before':>12} {'ECE after':>12}")
for t in ORDER:
    if t not in summary:
        continue
    s = summary[t]
    print(f"{t:<3} {s['temperature'][0]:>7.2f}+/-{s['temperature'][1]:.2f} "
          f"{s['nll_before'][0]:>12.5f} {s['nll_after'][0]:>12.5f} "
          f"{s['ece_before'][0]:>12.5f} {s['ece_after'][0]:>12.5f}")

print("\n" + "=" * W)
print("PAIRED SIGNIFICANCE TESTS (same seeds, two-sided paired t-test)")
print("=" * W)
print(f"{'Contrast':<10}{'Metric':<9}{'n':>3} {'mean diff':>11} {'95% CI':>22} {'p':>9} {'dz':>7}  verdict")
for c in contrasts:
    v = "significant" if c["p"] < 0.05 else "NOT significant"
    ci = f"[{c['ci95'][0]:+.2f}, {c['ci95'][1]:+.2f}]"
    dz = f"{c['cohen_dz']:+.2f}" if c["cohen_dz"] is not None else "  n/a"
    print(f"{c['contrast']:<10}{c['metric']:<9}{c['n']:>3} {c['mean_diff']:>+11.2f} {ci:>22} {c['p']:>9.4f} {dz:>7}  {v}")

print("\n" + "=" * W)
print("TEMPERATURE SCALING - calibration effect (paired, before vs after)")
print("=" * W)
for r in ts_stats:
    v = "significant" if r["p"] < 0.05 else "NOT significant"
    print(f"{r['config']}  {r['metric']:<8} n={r['n']}  {r['before']:.5f} -> {r['after']:.5f}  "
          f"({r['rel_change_pct']:+.1f}%)  p={r['p']:.4f}  {v}")

# ---------------- LaTeX ----------------
def esc(x):
    return x.replace("&", "\\&")


latex = []
latex.append("% Table 3 - ablation. Rank metrics are invariant to temperature scaling,")
latex.append("% so TS is reported in the calibration table only.")
latex.append("\\begin{table}[!t]")
latex.append("\\caption{Ablation study on the RoadObstacle21 validation split. Values are mean $\\pm$ standard")
latex.append(f"deviation over {len(SEEDS)} random seeds; each seed redraws the train/validation split and the")
latex.append("classifier initialisation. Arrows indicate the preferred direction.}")
latex.append("\\label{tab:ablation}")
latex.append("\\centering")
latex.append("\\begin{tabular}{@{}llccc@{}}")
latex.append("\\toprule")
latex.append("& Configuration & AuPRC $\\uparrow$ & FPR@95 $\\downarrow$ & IoU@0.5 $\\uparrow$ \\\\")
latex.append("\\midrule")
for t in [x for x in ("A", "B", "C") if x in summary]:
    s = summary[t]
    latex.append(f"{t} & {esc(s['name'])} & ${s['auprc'][0]:.2f} \\pm {s['auprc'][1]:.2f}$ & "
                 f"${s['fpr95'][0]:.3f} \\pm {s['fpr95'][1]:.3f}$ & ${s['iou_05'][0]:.2f} \\pm {s['iou_05'][1]:.2f}$ \\\\")
if "E" in summary or "F" in summary:
    latex.append("\\midrule")
    latex.append("\\multicolumn{5}{@{}l}{\\textit{Leave-one-out from the full model}} \\\\")
for t in [x for x in ("E", "F") if x in summary]:
    s = summary[t]
    latex.append(f"{t} & {esc(s['name'])} & ${s['auprc'][0]:.2f} \\pm {s['auprc'][1]:.2f}$ & "
                 f"${s['fpr95'][0]:.3f} \\pm {s['fpr95'][1]:.3f}$ & ${s['iou_05'][0]:.2f} \\pm {s['iou_05'][1]:.2f}$ \\\\")
latex.append("\\bottomrule")
latex.append("\\end{tabular}")
latex.append("\\end{table}")
latex.append("")
if "D" in summary:
    s = summary["D"]
    latex.append("\\begin{table}[!t]")
    latex.append("\\caption{Effect of temperature scaling on calibration of the full model. Ranking metrics")
    latex.append("(AuPRC, FPR@95, IoU) are mathematically invariant to a monotone rescaling of the logits and")
    latex.append("are therefore unchanged.}")
    latex.append("\\label{tab:calibration}")
    latex.append("\\centering")
    latex.append("\\begin{tabular}{@{}lccc@{}}")
    latex.append("\\toprule")
    latex.append("& NLL $\\downarrow$ & ECE-15 $\\downarrow$ & $T^{*}$ \\\\")
    latex.append("\\midrule")
    latex.append(f"Uncalibrated ($T=1$) & ${s['nll_before'][0]:.5f}$ & ${s['ece_before'][0]:.5f}$ & --- \\\\")
    latex.append(f"Temperature scaled & ${s['nll_after'][0]:.5f}$ & ${s['ece_after'][0]:.5f}$ & "
                 f"${s['temperature'][0]:.2f} \\pm {s['temperature'][1]:.2f}$ \\\\")
    latex.append("\\bottomrule")
    latex.append("\\end{tabular}")
    latex.append("\\end{table}")
tex = "\n".join(latex)
(ROOT / "table3.tex").write_text(tex)
print("\n" + "=" * W)
print("PASTE-READY LATEX (also saved to table3.tex)")
print("=" * W)
print(tex)

csvp = ROOT / "ablation_summary.csv"
cols = ["tag", "name", "cp", "sep", "ts", "auprc", "fpr95", "iou_05", "comp_det", "temperature", "nll_before", "nll_after", "ece_before", "ece_after"]
with csvp.open("w") as f:
    f.write("tag,name,cp,sep,ts," + ",".join(f"{c}_mean,{c}_sd" for c in cols[5:]) + "\n")
    for t in ORDER:
        if t not in summary:
            continue
        s = summary[t]
        f.write(f'{t},"{s["name"]}",{s["cp"]},{s["sep"]},{s["ts"]},'
                + ",".join(f"{s[c][0]:.6f},{s[c][1]:.6f}" for c in cols[5:]) + "\n")

if failures:
    print(f"\nWARNING: {failures} did not complete. Re-run this cell to retry only those.")

arch = Path(shutil.make_archive("/kaggle/working/obstacletrack_ablation_v3", "zip", root_dir=ROOT))
print("\nDownloads:")
for p in (ROOT / "ablation_summary.json", ROOT / "ablation_results_all_seeds.json", csvp, ROOT / "table3.tex", arch):
    display(FileLink(str(p)))


Seeds: [42, 1, 2, 3, 4]   Configs: ['A', 'B', 'C', 'D', 'E', 'F']
Each (config, seed) runs in a fresh process. D reuses C, F reuses B.
Trainings: 20  |  expect roughly 40-160 min depending on session speed


--- A seed=42 (attempt 1/2, timeout 30m) ---
TAG=A SEED=42 | Fine-tuned SegFormer-B0 (CE+Dice)


Loading weights: 100%|██████████| 208/208 [00:00<00:00, 1625.28it/s, Materializing param=segformer.encoder.patch_embeddings.3.proj.weight]            
SegformerForSemanticSegmentation LOAD REPORT from: /kaggle/working/segformer_b0_local
Key                           | Status   |                                                                                                   
------------------------------+----------+---------------------------------------------------------------------------------------------------
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([150, 256, 1, 1]) vs model:torch.Size([2, 256, 1, 1])
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([150]) vs model:torch.Size([2])                      

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


  epoch 01/12 val_loss=0.589671 (7s)
  epoch 02/12 val_loss=0.535272 (4s)
  epoch 03/12 val_loss=0.518288 (4s)
  epoch 04/12 val_loss=0.512889 (4s)
  epoch 05/12 val_loss=0.503950 (4s)
  epoch 06/12 val_loss=0.496473 (4s)
  epoch 07/12 val_loss=0.489518 (4s)
  epoch 08/12 val_loss=0.482821 (4s)
  epoch 09/12 val_loss=0.479660 (4s)
  epoch 10/12 val_loss=0.478318 (4s)
  epoch 11/12 val_loss=0.477398 (4s)
  epoch 12/12 val_loss=0.477995 (4s)


Loading weights: 100%|██████████| 208/208 [00:00<00:00, 1645.51it/s, Materializing param=segformer.encoder.patch_embeddings.3.proj.weight]            
SegformerForSemanticSegmentation LOAD REPORT from: /kaggle/working/segformer_b0_local
Key                           | Status   |                                                                                                   
------------------------------+----------+---------------------------------------------------------------------------------------------------
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([150, 256, 1, 1]) vs model:torch.Size([2, 256, 1, 1])
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([150]) vs model:torch.Size([2])                      

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


OK A seed=42 auprc=49.04 fpr95=1.348 iou=27.05 T=1.0 nll 0.02273->0.02273

--- B seed=42 (attempt 1/2, timeout 30m) ---
TAG=B SEED=42 | + Perspective-aware copy-paste
obstacle bank: 36 patches


...

/kaggle/working/ablation_v3/ablation_summary.json

/kaggle/working/ablation_v3/ablation_results_all_seeds.json

/kaggle/working/ablation_v3/ablation_summary.csv

/kaggle/working/ablation_v3/table3.tex

/kaggle/working/obstacletrack_ablation_v3.zip